In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import os
os.makedirs('/content/drive/MyDrive/VKR_TECD/output', exist_ok=True)
%cd /content/drive/MyDrive/VKR_TECD/
!pwd

/content/drive/MyDrive/VKR_TECD
/content/drive/MyDrive/VKR_TECD


In [3]:
!pip install -q pandas numpy scipy scikit-learn torch datasets huggingface_hub

In [5]:
import os
import pandas as pd
import numpy as np
from scipy.sparse import csr_matrix, save_npz
import pickle
import json
from collections import Counter
import glob
from huggingface_hub import snapshot_download

os.makedirs('data', exist_ok=True)
os.makedirs('output', exist_ok=True)

print("ШАГ 1: ЗАГРУЗКА И ПРЕДОБРАБОТКА T-ECD")

if not os.path.exists('data/dataset/small/marketplace'):
    print("Скачивание датасета T-ECD (может занять 10-15 минут)...")
    snapshot_download(
        repo_id="t-tech/T-ECD",
        repo_type="dataset",
        allow_patterns=[
            "dataset/small/marketplace/events/*.pq",
            "dataset/small/marketplace/items.pq",
        ],
        local_dir="./data",
        max_workers=2
    )
    print("Датасет скачан")
else:
    print("Датасет уже скачан")

events_path = "data/dataset/small/marketplace/events"
event_files = glob.glob(os.path.join(events_path, "*.pq"))
print(f"Найдено {len(event_files)} файлов событий")

REQUIRED_COLUMNS = ['user_id', 'item_id', 'action_type']
action_mapping = {'view': 1, 'click': 2, 'clickout': 3, 'like': 4}

print("ПРОХОД 1: Подсчет частот")
user_counter = Counter()
item_counter = Counter()
action_counter = Counter()
total_events = 0

for i, f in enumerate(event_files):
    if i % 50 == 0:
        print(f"  Обработано {i}/{len(event_files)} файлов")
    try:
        df = pd.read_parquet(f, columns=REQUIRED_COLUMNS)
        df = df.dropna(subset=['user_id', 'item_id', 'action_type'])
        if len(df) > 0:
            total_events += len(df)
            user_counter.update(df['user_id'].values)
            item_counter.update(df['item_id'].values)
            action_counter.update(df['action_type'].values)
    except Exception:
        continue

print(f"\nВсего событий: {total_events}")
print(f"Уникальных пользователей: {len(user_counter)}")
print(f"Уникальных товаров: {len(item_counter)}")
print("Типы взаимодействий:")
for action, count in action_counter.most_common():
    print(f"  {action}: {count}")

MIN_INTERACTIONS = 5
active_users = {uid for uid, cnt in user_counter.items() if cnt >= MIN_INTERACTIONS}
active_items = {iid for iid, cnt in item_counter.items() if cnt >= MIN_INTERACTIONS}

print(f"\nПосле фильтрации (>= {MIN_INTERACTIONS} взаимодействий):")
print(f"  Активных пользователей: {len(active_users)}")
print(f"  Активных товаров: {len(active_items)}")

print("\nПРОХОД 2: Построение матрицы")
user_ids = list(active_users)
item_ids = list(active_items)
user2idx = {uid: i for i, uid in enumerate(user_ids)}
item2idx = {iid: i for i, iid in enumerate(item_ids)}

rows_list, cols_list, data_list = [], [], []
interactions_count = 0

for i, f in enumerate(event_files):
    if i % 50 == 0:
        print(f"  Обработано {i}/{len(event_files)} файлов")
    try:
        df = pd.read_parquet(f, columns=REQUIRED_COLUMNS)
        df = df.dropna(subset=['user_id', 'item_id', 'action_type'])
        if len(df) > 0:
            mask = df['user_id'].isin(active_users) & df['item_id'].isin(active_items)
            df_filtered = df[mask].copy()
            if len(df_filtered) > 0:
                df_filtered['rating'] = df_filtered['action_type'].map(action_mapping)
                rows = df_filtered['user_id'].map(user2idx).values
                cols = df_filtered['item_id'].map(item2idx).values
                data = df_filtered['rating'].values.astype(np.float32)
                rows_list.append(rows)
                cols_list.append(cols)
                data_list.append(data)
                interactions_count += len(df_filtered)
    except Exception:
        continue

print(f"\nВсего взаимодействий в матрице: {interactions_count}")
print("Построение разреженной матрицы...")

all_rows = np.concatenate(rows_list)
all_cols = np.concatenate(cols_list)
all_data = np.concatenate(data_list)

R = csr_matrix((all_data, (all_rows, all_cols)),
               shape=(len(user_ids), len(item_ids)))

print(f"Размер матрицы: {R.shape}")
density = R.nnz / (R.shape[0] * R.shape[1])
print(f"Плотность матрицы: {density:.6%}")

print("\nСохранение результатов...")
np.save("output/user_ids.npy", np.array(user_ids))
np.save("output/item_ids.npy", np.array(item_ids))

with open("output/user2idx.pkl", "wb") as f:
    pickle.dump(user2idx, f)
with open("output/item2idx.pkl", "wb") as f:
    pickle.dump(item2idx, f)

save_npz("output/train_matrix.npz", R)

stats = {
    'total_events': total_events,
    'users_before': len(user_counter),
    'items_before': len(item_counter),
    'users_after': len(user_ids),
    'items_after': len(item_ids),
    'interactions_after': interactions_count,
    'matrix_shape': list(R.shape),
    'density': density,
    'action_types': dict(action_counter)
}

with open("output/statistics.json", "w", encoding="utf-8") as f:
    json.dump(stats, f, indent=2, ensure_ascii=False)

print("\nПРЕДОБРАБОТКА ЗАВЕРШЕНА")
print(f"Пользователей после фильтрации: {len(user_ids):,}")
print(f"Товаров после фильтрации: {len(item_ids):,}")
print(f"Взаимодействий: {interactions_count:,}")
print(f"Плотность матрицы: {density:.6%}")

ШАГ 1: ЗАГРУЗКА И ПРЕДОБРАБОТКА T-ECD
Датасет уже скачан
Найдено 227 файлов событий
ПРОХОД 1: Подсчет частот
  Обработано 0/227 файлов
  Обработано 50/227 файлов
  Обработано 100/227 файлов
  Обработано 150/227 файлов
  Обработано 200/227 файлов

Всего событий: 130447117
Уникальных пользователей: 1616763
Уникальных товаров: 2325409
Типы взаимодействий:
  view: 126147783
  click: 3772094
  clickout: 485448
  like: 41792

После фильтрации (>= 5 взаимодействий):
  Активных пользователей: 1382085
  Активных товаров: 760794

ПРОХОД 2: Построение матрицы
  Обработано 0/227 файлов
  Обработано 50/227 файлов
  Обработано 100/227 файлов
  Обработано 150/227 файлов
  Обработано 200/227 файлов

Всего взаимодействий в матрице: 127067374
Построение разреженной матрицы...
Размер матрицы: (1382085, 760794)
Плотность матрицы: 0.007899%

Сохранение результатов...

ПРЕДОБРАБОТКА ЗАВЕРШЕНА
Пользователей после фильтрации: 1,382,085
Товаров после фильтрации: 760,794
Взаимодействий: 127,067,374
Плотность ма